In [41]:
import numpy as np
from dataclasses import dataclass
from typing import Tuple, List, Dict
from collections import defaultdict
import itertools
import scipy.sparse as sp
import math 
import scipy.linalg as la
import scipy.sparse.linalg as sla
@dataclass
class TensorTerm:
    tensor: np.ndarray
    daggers: Tuple[int, ...]

class Superposition:
    """An object-oriented wrapper for a linear combination of Fock states."""
    def __init__(self, state_dict=None):
        self.states = defaultdict(complex, state_dict if state_dict else {})

    def __add__(self, other):
        result = self.states.copy()
        for st, amp in other.states.items():
            result[st] += amp
        return Superposition(self._clean_dict(result))

    def __rmul__(self, scalar):
        result = {st: scalar * amp for st, amp in self.states.items()}
        return Superposition(self._clean_dict(result))

    def _clean_dict(self, d, tol=1e-12):
        return {st: amp for st, amp in d.items() if abs(amp) > tol}

    def normalize(self):
        norm = np.sqrt(sum(abs(amp)**2 for amp in self.states.values()))
        if norm > 0:
            self.states = {st: amp / norm for st, amp in self.states.items()}
        return self

    def trim(self, threshold=0.99):
        self.normalize()
        sorted_items = sorted(self.states.items(), key=lambda item: abs(item[1])**2, reverse=True)
        trimmed_state = {}
        accumulated_prob = 0.0
        
        for st, amp in sorted_items:
            trimmed_state[st] = amp
            accumulated_prob += abs(amp)**2
            if accumulated_prob >= threshold:
                break
                
        self.states = trimmed_state
        return self.normalize()

class QuantumOperator:
    """An algebraically aware container for tensor terms."""
    def __init__(self, terms=None):
        self.terms = terms if terms is not None else []

    def __add__(self, other):
        return QuantumOperator(self.terms + other.terms)

    def __rmul__(self, scalar):
        new_terms = [TensorTerm(term.tensor * scalar, term.daggers) for term in self.terms]
        return QuantumOperator(new_terms)

    def __mul__(self, other):
        """Computes the product of two operators via tensor outer product."""
        new_terms = []
        for tA in self.terms:
            for tB in other.terms:
                # np.tensordot with axes=0 computes the full outer product
                new_tensor = np.tensordot(tA.tensor, tB.tensor, axes=0)
                new_daggers = tA.daggers + tB.daggers
                new_terms.append(TensorTerm(new_tensor, new_daggers))
        return QuantumOperator(new_terms)

    def simplify(self):
        """Optimizes the operator by merging tensors with identical dagger structures."""
        accumulated = {}
        for term in self.terms:
            if term.daggers not in accumulated:
                accumulated[term.daggers] = np.zeros_like(term.tensor)
            accumulated[term.daggers] += term.tensor
            
        self.terms = [
            TensorTerm(tensor, dags) 
            for dags, tensor in accumulated.items() 
            if np.any(np.abs(tensor) > 1e-12) # Drop pure noise tensors
        ]
        return self

In [52]:
class SecondQuantizationEngine:
    def __init__(self, n_modes: int, statistics: str = 'boson'):
        if statistics not in ['boson', 'fermion']:
            raise ValueError("Statistics must be 'boson' or 'fermion'")
        self.n_modes = n_modes
        self.statistics = statistics
        self.sign = 1 if statistics == 'boson' else -1  

    def _get_parity_phase(self, mode: int, state: Tuple[int, ...]) -> int:
        """Calculates the Jordan-Wigner phase for fermions."""
        if self.statistics == 'boson':
            return 1
        # For fermions, count particles in modes prior to the target mode
        particles_before = sum(state[:mode])
        return -1 if particles_before % 2 == 1 else 1

    def _apply_annihilation(self, mode: int, state: Tuple[int, ...]) -> Tuple[Tuple[int, ...], complex]:
        n = state[mode]
        if n == 0:
            return None, 0.0 # Annihilating the vacuum yields 0
        
        phase = self._get_parity_phase(mode, state)
        new_state = list(state)
        new_state[mode] -= 1
        
        amplitude = phase * np.sqrt(n)
        return tuple(new_state), amplitude

    def _apply_creation(self, mode: int, state: Tuple[int, ...]) -> Tuple[Tuple[int, ...], complex]:
        n = state[mode]
        if self.statistics == 'fermion' and n == 1:
            return None, 0.0 # Pauli exclusion principle
            
        phase = self._get_parity_phase(mode, state)
        new_state = list(state)
        new_state[mode] += 1
        
        amplitude = phase * np.sqrt(n + 1)
        return tuple(new_state), amplitude

    def apply_operator(self, operator: QuantumOperator, superposition: Superposition) -> Superposition:
        """Updated to interface seamlessly with the new Superposition class."""
        result_state = defaultdict(complex)
        active_states = superposition._clean_dict(superposition.states)
        
        if not active_states:
            return Superposition()
            
        for term in operator.terms:
            non_zero_indices = np.argwhere(term.tensor != 0)
            for idx in non_zero_indices:
                idx_tuple = tuple(idx)
                coeff = term.tensor[idx_tuple]
                
                current_branches = {st: amp * coeff for st, amp in active_states.items()}
                
                for i in reversed(range(len(term.daggers))):
                    mode = idx_tuple[i]
                    is_dagger = term.daggers[i]
                    next_branches = defaultdict(complex)
                    
                    for curr_state, curr_amp in current_branches.items():
                        if is_dagger:
                            new_st, amp_factor = self._apply_creation(mode, curr_state)
                        else:
                            new_st, amp_factor = self._apply_annihilation(mode, curr_state)
                            
                        if new_st is not None:
                            next_branches[new_st] += curr_amp * amp_factor
                            
                    current_branches = next_branches
                    if not current_branches:
                        break
                        
                for final_st, final_amp in current_branches.items():
                    result_state[final_st] += final_amp
                    
        return Superposition(result_state)
    
    def normalize_state(self, superposition: Dict[Tuple[int, ...], complex]) -> Dict[Tuple[int, ...], complex]:
        """
        Normalizes a linear combination of Fock states so that the sum of |c|^2 is 1.
        """
        # Calculate the norm: sqrt(sum(|c|^2))
        norm_sq = sum(abs(amp)**2 for amp in superposition.values())
        
        if np.isclose(norm_sq, 0.0):
            return {} # Cannot normalize the vacuum/zero-vector
            
        norm = np.sqrt(norm_sq)
        return {st: amp / norm for st, amp in superposition.items()}

    def trim_state(self, superposition: Dict[Tuple[int, ...], complex], threshold: float = 0.99) -> Dict[Tuple[int, ...], complex]:
        """
        Normalizes, sorts by magnitude descending, and truncates to keep the most 
        significant terms that sum up to the probability threshold.
        """
        # Step 1: Normalize the incoming state
        normalized_state = self.normalize_state(superposition)
        
        # Step 2: Sort descending by absolute magnitude squared (|c|^2)
        sorted_items = sorted(normalized_state.items(), key=lambda item: abs(item[1])**2, reverse=True)
        
        # Step 3: Accumulate probabilities and trim
        trimmed_state = {}
        accumulated_prob = 0.0
        
        for st, amp in sorted_items:
            prob = abs(amp)**2
            trimmed_state[st] = amp
            accumulated_prob += prob
            
            # Stop once the accumulated probability hits or exceeds the threshold
            if accumulated_prob >= threshold:
                break
                
        # Step 4: Re-normalize the trimmed state so it is completely unitary again
        return self.normalize_state(trimmed_state)
    
    def normal_order(self, operator: QuantumOperator) -> QuantumOperator:
        """
        Reduces a QuantumOperator to normal order using Wick's theorem.
        Returns a newly optimized QuantumOperator object.
        """
        pending_terms = operator.terms.copy()
        accumulated_terms = {}

        while pending_terms:
            term = pending_terms.pop(0)
            daggers = term.daggers
            tensor = np.asarray(term.tensor)
            
            swap_idx = -1
            for i in range(len(daggers) - 1):
                if daggers[i] == 0 and daggers[i+1] == 1:
                    swap_idx = i
                    break
                    
            if swap_idx != -1:
                # Swapped Term
                new_daggers_swap = list(daggers)
                new_daggers_swap[swap_idx], new_daggers_swap[swap_idx+1] = 1, 0
                new_tensor_swap = np.swapaxes(tensor, swap_idx, swap_idx+1) * self.sign
                pending_terms.append(TensorTerm(new_tensor_swap, tuple(new_daggers_swap)))
                
                # Contracted Term
                new_daggers_contract = tuple(daggers[:swap_idx] + daggers[swap_idx+2:])
                new_tensor_contract = np.trace(tensor, axis1=swap_idx, axis2=swap_idx+1)
                pending_terms.append(TensorTerm(new_tensor_contract, new_daggers_contract))
            else:
                if daggers not in accumulated_terms:
                    accumulated_terms[daggers] = np.zeros_like(tensor)
                accumulated_terms[daggers] += tensor
                
        final_terms = [
            TensorTerm(tensor, dags) 
            for dags, tensor in accumulated_terms.items() 
            if not np.allclose(tensor, 0, atol=1e-12)
        ]
        return QuantumOperator(final_terms)
    
    def change_basis(self, operator: QuantumOperator, U: np.ndarray, V: np.ndarray) -> QuantumOperator:
        """
        Applies a Bogoliubov transformation to the operator.
        b_i = U_ij b_j + V*_ij b^dag_j
        """
        new_terms = []
        
        for original_term in operator.terms:
            k = len(original_term.daggers)
            
            # Start the branching with the current tensor and daggers
            # We use lists for daggers temporarily so we can mutate them during branching
            branches = [(original_term.tensor, list(original_term.daggers))]
            
            for axis in range(k):
                next_branches = []
                
                for tensor, daggers in branches:
                    is_dagger = daggers[axis]
                    
                    if is_dagger == 0:
                        # Annihilation transforms: b_i = U_ij b_j + V*_ij b^dag_j
                        
                        # Branch 1: U (annihilation)
                        t_U = np.tensordot(tensor, U, axes=([axis], [0]))
                        t_U = np.moveaxis(t_U, -1, axis)
                        dag_U = daggers.copy()
                        dag_U[axis] = 0
                        next_branches.append((t_U, dag_U))
                        
                        # Branch 2: V^* (creation)
                        t_V_conj = np.tensordot(tensor, np.conj(V), axes=([axis], [0]))
                        t_V_conj = np.moveaxis(t_V_conj, -1, axis)
                        dag_V = daggers.copy()
                        dag_V[axis] = 1
                        next_branches.append((t_V_conj, dag_V))
                        
                    else:
                        # Creation transforms: b^dag_i = U*_ij b^dag_j + V_ij b_j
                        
                        # Branch 1: U^* (creation)
                        t_U_conj = np.tensordot(tensor, np.conj(U), axes=([axis], [0]))
                        t_U_conj = np.moveaxis(t_U_conj, -1, axis)
                        dag_U_conj = daggers.copy()
                        dag_U_conj[axis] = 1
                        next_branches.append((t_U_conj, dag_U_conj))
                        
                        # Branch 2: V (annihilation)
                        t_V = np.tensordot(tensor, V, axes=([axis], [0]))
                        t_V = np.moveaxis(t_V, -1, axis)
                        dag_V = daggers.copy()
                        dag_V[axis] = 0
                        next_branches.append((t_V, dag_V))
                        
                # Update branches for the next axis
                branches = next_branches
                
            # After iterating all axes, commit the 2^k branches to the new operator
            for tensor, dags in branches:
                if not np.allclose(tensor, 0): # Filter out exact zeros to save memory
                    new_terms.append(TensorTerm(tensor, tuple(dags)))
                    
        return QuantumOperator(new_terms)

    def generate_basis(self, n_excitations: int) -> List[Tuple[int, ...]]:
        """
        Generates all valid Fock states for the system with exactly `n_excitations`.
        Automatically handles Bosonic vs Fermionic statistics.
        """
        if n_excitations < 0:
            return []
            
        if self.statistics == 'fermion':
            if n_excitations > self.n_modes:
                return []  # Pauli exclusion: cannot have more fermions than modes
                
            # For fermions, we simply choose `n_excitations` modes to occupy
            basis = []
            for occupied_indices in itertools.combinations(range(self.n_modes), n_excitations):
                state = [0] * self.n_modes
                for idx in occupied_indices:
                    state[idx] = 1
                basis.append(tuple(state))
            return basis
            
        else:
            # For bosons, we use the recursive stars-and-bars integer partitioning
            def compositions(n, total):
                if n == 1:
                    yield (total,)
                    return
                for first in range(total + 1):
                    for rest in compositions(n - 1, total - first):
                        yield (first,) + rest
                        
            return list(compositions(self.n_modes, n_excitations))

    def build_operator_matrix(self, operator: QuantumOperator, basis: List[Tuple[int, ...]]) -> np.ndarray:
        """
        Computes the dense matrix representation of an operator within a given basis.
        Rows correspond to the 'bra' (output state), Columns correspond to the 'ket' (input state).
        """
        dim = len(basis)
        matrix = np.zeros((dim, dim), dtype=complex)
        
        # Create a fast lookup dictionary for the row index of each resulting state
        state_to_index = {state: idx for idx, state in enumerate(basis)}
        
        # Iterate over the basis (Columns/Kets)
        for col_idx, ket_state in enumerate(basis):
            # Formulate the ket as a superposition dictionary
            initial_superposition = {ket_state: 1.0 + 0j}
            
            # Apply the operator to the ket
            result_superposition = self.apply_operator(operator, initial_superposition)
            
            # Populate the matrix elements (Rows/Bras)
            for result_state, amplitude in result_superposition.items():
                if result_state in state_to_index:
                    row_idx = state_to_index[result_state]
                    matrix[row_idx, col_idx] += amplitude
                    
        return matrix

    def build_sparse_operator_matrix(self, operator: QuantumOperator, basis: List[Tuple[int, ...]]) -> sp.csr_matrix:
        """
        Computes the sparse matrix representation of an operator within a given basis.
        Returns a CSR format sparse matrix optimized for fast arithmetic and diagonalization.
        """
        dim = len(basis)
        
        # DOK (Dictionary of Keys) format is the fastest for constructing sparse matrices 
        # when you are adding elements one by one.
        matrix = sp.dok_matrix((dim, dim), dtype=complex)
        
        # Fast lookup dictionary
        state_to_index = {state: idx for idx, state in enumerate(basis)}
        
        # Iterate over the basis (Columns/Kets)
        for col_idx, ket_state in enumerate(basis):
            initial_superposition = {ket_state: 1.0 + 0j}
            
            # Apply the operator to the ket
            result_superposition = self.apply_operator(operator, initial_superposition)
            
            # Populate the matrix elements (Rows/Bras)
            for result_state, amplitude in result_superposition.items():
                if result_state in state_to_index:
                    row_idx = state_to_index[result_state]
                    matrix[row_idx, col_idx] += amplitude
                    
        # Convert to CSR format before returning. 
        # CSR is required for fast Scipy linear algebra operations (like eigsh).
        return matrix.tocsr()

    def build_fast_sparse_matrix(self, operator: QuantumOperator, basis: list) -> sp.csr_matrix:
        """Massively optimized sparse matrix builder using COO format."""
        dim = len(basis)
        state_to_index = {state: idx for idx, state in enumerate(basis)}
        
        operator.simplify() # Ensure redundant tensors are merged before loop
        
        rows, cols, data = [], [], []
        
        for col_idx, ket_state in enumerate(basis):
            initial_super = Superposition({ket_state: 1.0 + 0j})
            result_super = self.apply_operator(operator, initial_super)
            
            for result_state, amplitude in result_super.states.items():
                if result_state in state_to_index:
                    rows.append(state_to_index[result_state])
                    cols.append(col_idx)
                    data.append(amplitude)
                    
        # Construct the COO matrix in a single vectorized burst, then convert to CSR
        coo = sp.coo_matrix((data, (rows, cols)), shape=(dim, dim), dtype=complex)
        return coo.tocsr()
    
    def exp_operator(self, operator: QuantumOperator, order: int) -> QuantumOperator:
        """
        Computes the Taylor expansion exp(A) = I + A + A^2/2! + ... + A^n/n!
        and returns the fully normal-ordered QuantumOperator.
        """
        # Identity operator: 0-order tensor, scalar 1.0
        identity_term = TensorTerm(np.array(1.0 + 0j), ())
        result_op = QuantumOperator([identity_term])
        
        if order == 0:
            return result_op
            
        current_power = operator
        result_op = result_op + (1.0 * current_power)
        
        for n in range(2, order + 1):
            # Multiply operator by itself to get next power
            current_power = current_power * operator
            
            # Crucial optimization: simplify before the next multiplication 
            # to prevent the tensor array sizes from bottlenecking RAM
            current_power.simplify() 
            
            term_to_add = (1.0 / math.factorial(n)) * current_power
            result_op = result_op + term_to_add
            
        result_op.simplify()
        
        # Pass the final summation through the Wick's theorem engine
        return self.normal_order(result_op)

    def _extract_quadratic_blocks(self, operator: QuantumOperator) -> Tuple[np.ndarray, np.ndarray]:
        """Extracts the A (b^dag b) and B (b^dag b^dag) matrices from a quadratic operator."""
        operator.simplify()
        A = np.zeros((self.n_modes, self.n_modes), dtype=complex)
        B = np.zeros((self.n_modes, self.n_modes), dtype=complex)
        
        for term in operator.terms:
            if len(term.daggers) != 2:
                raise ValueError("Operator contains non-quadratic terms.")
                
            if term.daggers == (1, 0):    # b^dag_i b_j
                A += term.tensor
            elif term.daggers == (1, 1):  # b^dag_i b^dag_j
                B += term.tensor
                
        return A, B

    def diagonalize_quadratic(self, operator: QuantumOperator):
        # ... (extract A and B)
        A, B = self._extract_quadratic_blocks(operator)
        n = self.n_modes
        
        # FIX: Multiply B by 2 to align the Van Hemmen 1/2 factor
        # with standard tensor summation conventions
        B_scaled = 2.0 * B 
        
        # Build the Hermitian dynamical matrix
        D_herm = np.block([
            [A, B_scaled],
            [B_scaled.conj().T, A.conj()]
        ])
        
        tau_z = np.block([
            [np.eye(n), np.zeros((n, n))],
            [np.zeros((n, n)), -np.eye(n)]
        ])
        
        # Van Hemmen Cholesky decomposition
        L = np.linalg.cholesky(D_herm)
        M_sym = L.T @ tau_z @ L
        
        evals_sym, evecs_sym = la.eigh(M_sym)
        evecs_real = la.solve_triangular(L.T, evecs_sym)
        
        # Keep positive branch
        pos_idx = evals_sym > 1e-15
        pos_evals = evals_sym[pos_idx]
        pos_evecs = evecs_real[:, pos_idx]
        
        sort_idx = np.argsort(pos_evals)
        pos_evals = pos_evals[sort_idx]
        pos_evecs = pos_evecs[:, sort_idx]
        
        # Symplectic Gram-Schmidt
        def symplectic_gram_schmidt(vecs):
            n_cols = vecs.shape[1]
            result = np.zeros_like(vecs, dtype=complex)
            for i in range(n_cols):
                v = vecs[:, i].copy()
                for j in range(i):
                    e_j = result[:, j]
                    proj = e_j.conj().T @ tau_z @ v
                    v = v - proj * e_j
                norm = np.real(v.conj().T @ tau_z @ v)
                result[:, i] = v / np.sqrt(norm)
            return result
            
        U_V = symplectic_gram_schmidt(pos_evecs)
        
        U = U_V[:n, :]
        V = U_V[n:, :]
        
        return pos_evals, U, V
    
    def build_squeezed_vacuum(self, U: np.ndarray, V: np.ndarray, truncation: int, threshold: float = 0.99) -> Superposition:
        """
        Generates the exact analytical squeezed vacuum state for a quadratic Hamiltonian.
        """
        n = self.n_modes
        
        # RECONSTRUCT U_V HERE:
        U_V = np.vstack([U, V])
        
        # Build the full pseudo-unitary transformation matrix to find the inverse
        tau_x = np.block([[np.zeros((n, n)), np.eye(n)], [np.eye(n), np.zeros((n, n))]])
        J_U_V = tau_x @ U_V.conj() # Pseudo-time reversal for the negative branch
        P = np.hstack([U_V, J_U_V])
        P_inv = np.linalg.inv(P)
        
        U_tilde = P_inv[:n, :n]
        V_tilde = P_inv[:n, n:]
        M = np.linalg.solve(U_tilde, V_tilde)
        
        vacuum = tuple([0] * n)
        current_terms = {vacuum: 1.0 + 0j}

        final_state = defaultdict(complex)
        final_state[vacuum] = 1.0 + 0j
        
        for k in range(1, truncation + 1):
            next_terms = defaultdict(complex)
            for state_tuple, coeff in current_terms.items():
                state = list(state_tuple)
                for j in range(n):
                    for i in range(n):
                        val = M[i, j]
                        if abs(val) < 1e-9:
                            continue
                            
                        new_state = state.copy()
                        new_state[j] += 1
                        factor_j = np.sqrt(new_state[j])
                        new_state[i] += 1
                        factor_i = np.sqrt(new_state[i])
                        
                        next_terms[tuple(new_state)] += coeff * (-0.5) * val * factor_j * factor_i / float(k)
                        
            for st, val in next_terms.items():
                if abs(val) > 1e-9:
                    final_state[st] += val
            current_terms = next_terms
            
        return Superposition(final_state).trim(threshold)
    
    def transform_state(self, state: Superposition, U: np.ndarray, V: np.ndarray, direction: str = 'old_to_new') -> Superposition:
        """
        Transforms a state vector between the bare Fock basis and the Bogoliubov eigenbasis.
        """
        n = self.n_modes
        if direction == 'old_to_new':
            U_mat, V_mat = U, V
            base_vacuum = self.build_squeezed_vacuum(U, V, truncation=10) # Approximated base
        else:
            # Inverse mapping
            P = np.block([[U, V.conj()], [V, U.conj()]])
            P_inv = np.linalg.inv(P)
            U_mat, V_mat = P_inv[:n, :n], -P_inv[n:, :n]
            base_vacuum = Superposition({tuple([0] * n): 1.0 + 0j})
            
        result = Superposition()
        
        for st_tuple, amp in state.states.items():
            occupations = np.array(st_tuple)
            current_branch = base_vacuum
            
            for mode_idx in range(n):
                r_i = occupations[mode_idx]
                if r_i == 0:
                    continue
                    
                for k in range(1, r_i + 1):
                    # Apply the Bogoliubov transformed creation operator
                    next_branch = defaultdict(complex)
                    for br_st, br_amp in current_branch.states.items():
                        # U component (Creation)
                        for j in range(n):
                            if abs(U_mat[mode_idx, j]) > 1e-9:
                                new_st, factor = self._apply_creation(j, br_st)
                                if new_st:
                                    next_branch[new_st] += br_amp * U_mat[mode_idx, j] * factor
                        # V component (Annihilation)
                        for j in range(n):
                            if abs(V_mat[mode_idx, j]) > 1e-9:
                                new_st, factor = self._apply_annihilation(j, br_st)
                                if new_st:
                                    next_branch[new_st] += br_amp * V_mat[mode_idx, j] * factor
                                    
                    # Normalize factorial scaling
                    current_branch = Superposition({s: v / np.sqrt(k) for s, v in next_branch.items()})
                    
            result = result + (amp * current_branch)
            
        return result.trim()
    
    def build_quadratic_sparse_matrix(self, operator: QuantumOperator, basis: list) -> sp.csr_matrix:
        """
        Highly optimized sparse matrix builder specifically for quadratic Hamiltonians.
        Bypasses generalized tensor contractions for massive speedup.
        """
        A, B = self._extract_quadratic_blocks(operator)
        n = self.n_modes
        dim = len(basis)
        state_to_index = {state: idx for idx, state in enumerate(basis)}
        
        rows, cols, data = [], [], []
        
        def add_element(r, c, val):
            if abs(val) > 1e-9:
                rows.append(r)
                cols.append(c)
                data.append(val)
                
        for col_idx, ket in enumerate(basis):
            ket_arr = np.array(ket)
            
            # Number-conserving terms: A (b^dag_i b_j)
            for j in range(n):
                if ket_arr[j] == 0:
                    continue
                for i in range(n):
                    if abs(A[i, j]) < 1e-9:
                        continue
                    new_st = ket_arr.copy()
                    new_st[j] -= 1
                    factor_j = np.sqrt(ket_arr[j])
                    factor_i = np.sqrt(new_st[i] + 1)
                    new_st[i] += 1
                    
                    st_tuple = tuple(new_st)
                    if st_tuple in state_to_index:
                        add_element(state_to_index[st_tuple], col_idx, A[i, j] * factor_i * factor_j)
                        
            # Pairing terms: B (b^dag_i b^dag_j)
            for j in range(n):
                for i in range(n):
                    if abs(B[i, j]) < 1e-9:
                        continue
                    new_st = ket_arr.copy()
                    new_st[j] += 1
                    factor_j = np.sqrt(new_st[j])
                    new_st[i] += 1
                    factor_i = np.sqrt(new_st[i])
                    
                    st_tuple = tuple(new_st)
                    if st_tuple in state_to_index:
                        add_element(state_to_index[st_tuple], col_idx, B[i, j] * factor_i * factor_j)
                        
            # Conjugate Pairing terms: B^* (b_i b_j)
            B_conj = B.conj().T
            for j in range(n):
                if ket_arr[j] == 0:
                    continue
                for i in range(n):
                    if abs(B_conj[i, j]) < 1e-9:
                        continue
                    new_st = ket_arr.copy()
                    factor_j = np.sqrt(new_st[j])
                    new_st[j] -= 1
                    if new_st[i] == 0:
                        continue
                    factor_i = np.sqrt(new_st[i])
                    new_st[i] -= 1
                    
                    st_tuple = tuple(new_st)
                    if st_tuple in state_to_index:
                        add_element(state_to_index[st_tuple], col_idx, B_conj[i, j] * factor_i * factor_j)

        return sp.coo_matrix((data, (rows, cols)), shape=(dim, dim), dtype=complex).tocsr()
    
    def partial_trace(self, state: Superposition, keep_modes: List[int]) -> np.ndarray:
        """
        Computes the reduced density matrix by tracing out all modes NOT in `keep_modes`.
        """
        keep_modes = sorted(keep_modes)
        trace_modes = [i for i in range(self.n_modes) if i not in keep_modes]
        
        # Determine the dimension of the kept subspace dynamically based on the state
        max_excitations = {m: 0 for m in keep_modes}
        for st in state.states.keys():
            for m in keep_modes:
                max_excitations[m] = max(max_excitations[m], st[m])
                
        # Create an index mapping for the kept basis
        import itertools
        ranges = [range(max_excitations[m] + 1) for m in keep_modes]
        kept_basis = list(itertools.product(*ranges))
        kept_idx = {st: i for i, st in enumerate(kept_basis)}
        dim = len(kept_basis)
        
        rho_reduced = np.zeros((dim, dim), dtype=complex)
        
        # Group states by their traced-out mode configuration
        trace_groups = defaultdict(list)
        for st_tuple, amp in state.states.items():
            traced_config = tuple(st_tuple[m] for m in trace_modes)
            kept_config = tuple(st_tuple[m] for m in keep_modes)
            trace_groups[traced_config].append((kept_config, amp))
            
        # Perform the partial trace: sum over outer products of matching traced configurations
        for traced_config, kept_components in trace_groups.items():
            for (kept_1, amp_1) in kept_components:
                for (kept_2, amp_2) in kept_components:
                    i = kept_idx[kept_1]
                    j = kept_idx[kept_2]
                    rho_reduced[i, j] += amp_1 * np.conj(amp_2)
                    
        return rho_reduced
    
    def get_mode_occupations(self, state: Superposition) -> np.ndarray:
        """Calculates the expected photon/excitation number <n_i> for each mode."""
        occupations = np.zeros(self.n_modes, dtype=float)
        for st_tuple, amp in state.states.items():
            prob = abs(amp)**2
            for i in range(self.n_modes):
                occupations[i] += st_tuple[i] * prob
        return occupations

    def get_total_excitations(self, state: Superposition) -> float:
        """Returns the total expected number of excitations in the system."""
        return np.sum(self.get_mode_occupations(state))
    

class CompositeEngine:
    """
    Handles the tensor products of states and operators for bipartite quantum systems,
    such as light-matter interfaces (Boson-Fermion composite systems).
    """
    def __init__(self, engine_A: SecondQuantizationEngine, engine_B: SecondQuantizationEngine):
        self.engine_A = engine_A
        self.engine_B = engine_B

    def tensor_states(self, state_A: Superposition, state_B: Superposition) -> Superposition:
        """
        Computes the Kronecker product of two state vectors |Psi_A> ⊗ |Psi_B>.
        Returns a new Superposition in the composite Fock basis.
        """
        result_states = {}
        for st_A, amp_A in state_A.states.items():
            for st_B, amp_B in state_B.states.items():
                # Concatenate the tuples to form the composite state
                composite_st = st_A + st_B
                result_states[composite_st] = amp_A * amp_B
                
        return Superposition(result_states).trim(threshold=1.0) # threshold=1.0 keeps all terms

    def generate_composite_basis(self, basis_A: list, basis_B: list) -> list:
        """Creates the full ordered basis list for the composite Hilbert space."""
        import itertools
        return [st_A + st_B for st_A, st_B in itertools.product(basis_A, basis_B)]

    def embed_matrix_A(self, matrix_A: sp.csr_matrix, dim_B: int) -> sp.csr_matrix:
        """
        Embeds an operator from subsystem A into the full Hilbert space: H_A ⊗ I_B.
        """
        I_B = sp.eye(dim_B, format='csr')
        return sp.kron(matrix_A, I_B, format='csr')

    def embed_matrix_B(self, dim_A: int, matrix_B: sp.csr_matrix) -> sp.csr_matrix:
        """
        Embeds an operator from subsystem B into the full Hilbert space: I_A ⊗ H_B.
        """
        I_A = sp.eye(dim_A, format='csr')
        return sp.kron(I_A, matrix_B, format='csr')

    def kron_matrices(self, matrix_A: sp.csr_matrix, matrix_B: sp.csr_matrix) -> sp.csr_matrix:
        """
        Computes the Kronecker product of two operators: O_A ⊗ O_B.
        Used primarily for interaction Hamiltonians.
        """
        return sp.kron(matrix_A, matrix_B, format='csr')

In [50]:

# 1. Initialize Engine (2 modes, Bosonic)
engine = SecondQuantizationEngine(n_modes=2, statistics='boson')

# 2. Define the Two-Mode Squeezing Hamiltonian
omega = 1.0  # Bare mode energies
g = 0.6      # Squeezing/Coupling strength

# A matrix terms: \omega (a^\dagger a + b^\dagger b)
tensor_A = np.zeros((2, 2), dtype=complex)
tensor_A[0, 0] = omega
tensor_A[1, 1] = omega

# B matrix terms: g * a^\dagger b^\dagger
tensor_B = np.zeros((2, 2), dtype=complex)
tensor_B[0, 1] = 0.5 * g
tensor_B[1, 0] = 0.5 * g

# Conjugate B terms: g * a b
tensor_B_conj = np.zeros((2, 2), dtype=complex)
tensor_B_conj[0, 1] = 0.5 * g
tensor_B_conj[1, 0] = 0.5 * g

# Construct Operator
H_squeezer = QuantumOperator([
    TensorTerm(tensor_A, daggers=(1, 0)),
    TensorTerm(tensor_B, daggers=(1, 1)),
    TensorTerm(tensor_B_conj, daggers=(0, 0))
])


# --- TEST 1: Symplectic Diagonalization ---
print("--- 1. Symplectic Diagonalization (Bogoliubov) ---")
pos_evals, U, V = engine.diagonalize_quadratic(H_squeezer)

print("Bare Energies:       [1.0, 1.0]")
print(f"Bogoliubov Energies: [{pos_evals[0]:.4f}, {pos_evals[1]:.4f}]")
# Analytically, eigenvalues should be sqrt(omega^2 - g^2) = sqrt(1.0 - 0.36) = 0.8


# --- TEST 2: Analytical Squeezed Vacuum Generation ---
print("\n--- 2. Analytical Squeezed Vacuum ---")
# Generate the exact ground state up to 6 photon pairs
squeezed_vac = engine.build_squeezed_vacuum(U, V, truncation=6, threshold=0.999)

for st, amp in squeezed_vac.states.items():
    prob = abs(amp)**2
    print(f"State {st}: Amplitude = {amp.real:+.4f}, Probability = {prob:.4f}")


# --- TEST 3: State Transformation (Ket Basis Change) ---
print("\n--- 3. State Transformation ---")
# If we take the bare vacuum |0,0> and transform it to the new basis, 
# it should perfectly match the squeezed vacuum we just generated!
bare_vacuum = Superposition({(0, 0): 1.0 + 0j})
transformed_vac = engine.transform_state(bare_vacuum, U, V, direction='old_to_new')

# Let's print the first few terms to verify it matches Test 2
for st in [(0,0), (1,1), (2,2)]:
    if st in transformed_vac.states:
         print(f"Transformed State {st}: Amplitude = {transformed_vac.states[st].real:+.4f}")


# --- TEST 4: Fast Quadratic Sparse Matrix Builder ---
print("\n--- 4. Fast Sparse Matrix Diagonalization ---")
# Generate basis up to 6 total excitations
basis = []
for excitations in range(7):
    basis.extend(engine.generate_basis(n_excitations=excitations))

# Build sparse matrix directly from quadratic A and B blocks
sparse_H = engine.build_quadratic_sparse_matrix(H_squeezer, basis)
print(f"Sparse Matrix built: {sparse_H.shape} dimensions, {sparse_H.nnz} non-zero elements.")

# Use SciPy's Lanczos algorithm to find the lowest energy eigenvalue
# (We shift-invert 'SA' to find the smallest algebraic eigenvalue)
evals_num, evecs_num = sla.eigsh(sparse_H, k=1, which='SA')

# Calculate Analytical Ground State Energy (Zero Point Energy Shift)
# E_G = 0.5 * sum(Bogoliubov Energies) - 0.5 * sum(Bare Energies)
E_G_analytical = 0.5 * np.sum(pos_evals) - 0.5 * (omega + omega)

print(f"Numerical Ground State Energy:  {evals_num[0]:.6f}")
print(f"Analytical Ground State Energy: {E_G_analytical:.6f}")

--- 1. Symplectic Diagonalization (Bogoliubov) ---
Bare Energies:       [1.0, 1.0]
Bogoliubov Energies: [0.8000, 0.8000]

--- 2. Analytical Squeezed Vacuum ---
State (0, 0): Amplitude = +0.9429, Probability = 0.8890
State (1, 1): Amplitude = -0.3143, Probability = 0.0988
State (2, 2): Amplitude = +0.1048, Probability = 0.0110
State (3, 3): Amplitude = -0.0349, Probability = 0.0012

--- 3. State Transformation ---
Transformed State (0, 0): Amplitude = +0.9435
Transformed State (1, 1): Amplitude = -0.3145
Transformed State (2, 2): Amplitude = +0.1048

--- 4. Fast Sparse Matrix Diagonalization ---
Sparse Matrix built: (28, 28) dimensions, 57 non-zero elements.
Numerical Ground State Energy:  -0.199171
Analytical Ground State Energy: -0.200000


In [51]:
# --- TEST 5: Observables ---
print("\n--- 5. Observables ---")
occupations = engine.get_mode_occupations(squeezed_vac)
print(f"Mode Occupations: <n_a> = {occupations[0]:.4f}, <n_b> = {occupations[1]:.4f}")
print(f"Total Excitations in the Vacuum: {engine.get_total_excitations(squeezed_vac):.4f}")


# --- TEST 6: Partial Trace & Entanglement ---
print("\n--- 6. Partial Trace & Entanglement ---")
# Trace out mode 1 (b), keep only mode 0 (a)
rho_a = engine.partial_trace(squeezed_vac, keep_modes=[0])

# Calculate Purity: Tr(rho^2)
# If purity == 1, the state is pure (unentangled).
# If purity < 1, the state is mixed (entangled with the traced-out environment).
purity = np.trace(rho_a @ rho_a).real

print(f"Purity of Mode 'a' Reduced Density Matrix: {purity:.4f}")
if purity < 0.999:
    print("Result: Purity < 1.0! The modes are definitively entangled.")
    
# Let's look at the diagonal of the reduced density matrix
# It should perfectly match the thermal probabilities we saw in Test 2!
print("Diagonal elements of rho_a (Thermal Probabilities):")
for i in range(min(4, rho_a.shape[0])):
    print(f"  P({i} photons) = {rho_a[i, i].real:.4f}")


--- 5. Observables ---
Mode Occupations: <n_a> = 0.1244, <n_b> = 0.1244
Total Excitations in the Vacuum: 0.2488

--- 6. Partial Trace & Entanglement ---
Purity of Mode 'a' Reduced Density Matrix: 0.8002
Result: Purity < 1.0! The modes are definitively entangled.
Diagonal elements of rho_a (Thermal Probabilities):
  P(0 photons) = 0.8890
  P(1 photons) = 0.0988
  P(2 photons) = 0.0110
  P(3 photons) = 0.0012


In [53]:
# --- 1. Initialize Subsystems ---
# Cavity: 1 Bosonic mode
engine_cavity = SecondQuantizationEngine(n_modes=1, statistics='boson')
# Qubit/Atom: 1 Fermionic mode (acts exactly as a Pauli 2-level system)
engine_qubit = SecondQuantizationEngine(n_modes=1, statistics='fermion')

composite = CompositeEngine(engine_cavity, engine_qubit)

# --- 2. Parameters & Bases ---
omega_c = 1.0  # Cavity frequency
omega_q = 1.0  # Qubit frequency (On resonance)
g = 0.1        # Light-matter coupling strength

# Truncate cavity at 3 photons. Qubit naturally truncates at 1 excitation (Fermion)
basis_cavity = engine_cavity.generate_basis(0) + engine_cavity.generate_basis(1) + \
               engine_cavity.generate_basis(2) + engine_cavity.generate_basis(3)
basis_qubit = engine_qubit.generate_basis(0) + engine_qubit.generate_basis(1)

dim_c = len(basis_cavity)
dim_q = len(basis_qubit)

print(f"--- 1. Composite Hilbert Space ---")
print(f"Cavity Dimension: {dim_c}, Qubit Dimension: {dim_q}")
print(f"Composite Dimension: {dim_c * dim_q}")


# --- 3. Build Isolated Operators ---
# Cavity Number Operator: a^\dagger a
op_num_c = QuantumOperator([TensorTerm(np.array([[1.0]]), daggers=(1, 0))])
# Cavity Annihilation: a
op_a = QuantumOperator([TensorTerm(np.array([1.0]), daggers=(0,))])
# Cavity Creation: a^\dagger
op_adag = QuantumOperator([TensorTerm(np.array([1.0]), daggers=(1,))])

# Qubit Number Operator: c^\dagger c (sigma^+ sigma^-)
op_num_q = QuantumOperator([TensorTerm(np.array([[1.0]]), daggers=(1, 0))])
# Qubit Annihilation: c (sigma^-)
op_sm = QuantumOperator([TensorTerm(np.array([1.0]), daggers=(0,))])
# Qubit Creation: c^\dagger (sigma^+)
op_sp = QuantumOperator([TensorTerm(np.array([1.0]), daggers=(1,))])

# Convert to sparse matrices in their respective isolated bases
mat_num_c = engine_cavity.build_fast_sparse_matrix(op_num_c, basis_cavity)
mat_a     = engine_cavity.build_fast_sparse_matrix(op_a, basis_cavity)
mat_adag  = engine_cavity.build_fast_sparse_matrix(op_adag, basis_cavity)

mat_num_q = engine_qubit.build_fast_sparse_matrix(op_num_q, basis_qubit)
mat_sm    = engine_qubit.build_fast_sparse_matrix(op_sm, basis_qubit)
mat_sp    = engine_qubit.build_fast_sparse_matrix(op_sp, basis_qubit)


# --- 4. Build the Full Composite Hamiltonian ---
# H_bare = (w_c * a^dag a) ⊗ I_q  +  I_c ⊗ (w_q * c^dag c)
H_bare = composite.embed_matrix_A(omega_c * mat_num_c, dim_q) + \
         composite.embed_matrix_B(dim_c, omega_q * mat_num_q)

# H_int = g * (a^dag ⊗ sigma^-  +  a ⊗ sigma^+)
H_int = g * (composite.kron_matrices(mat_adag, mat_sm) + \
             composite.kron_matrices(mat_a, mat_sp))

H_total = H_bare + H_int


# --- 5. Diagonalize and Observe Vacuum Rabi Splitting ---
print("\n--- 2. Jaynes-Cummings Spectrum ---")
# Find the lowest 4 eigenvalues
evals, evecs = sla.eigsh(H_total, k=4, which='SA')

# Print the energies. We expect:
# E_0 = 0.0 (Ground State: |0, g>)
# E_1 = 1.0 - g = 0.9 (Lower Polariton: |1, g> - |0, e>)
# E_2 = 1.0 + g = 1.1 (Upper Polariton: |1, g> + |0, e>)
for i, E in enumerate(evals):
    print(f"Energy Level {i}: {E:.4f}")

--- 1. Composite Hilbert Space ---
Cavity Dimension: 4, Qubit Dimension: 2
Composite Dimension: 8

--- 2. Jaynes-Cummings Spectrum ---
Energy Level 0: 0.9000
Energy Level 1: 1.1000
Energy Level 2: 1.8586
Energy Level 3: 0.0000
